In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [2]:
df = pd.read_parquet("/Users/jasonc/Desktop/DSC_291/merged_features.parquet")

In [3]:
df

,total_amount,mean_amount,std_amount,min_amount,max_amount,median_amount,transaction_count,credit_sum,debit_sum,credit_debit_ratio,...,fft_high_freq_power_cat31.0,fft_power_ratio_cat31.0,fft_dominant_freq_cat33.0,fft_dominant_power_cat33.0,fft_spectral_entropy_cat33.0,fft_low_freq_power_cat33.0,fft_high_freq_power_cat33.0,fft_power_ratio_cat33.0,dataset_type,FPF_TARGET
masked_consumer_id,,,,,,,,,,,,,,,,,,,,,
C01100001,-8111.42,-3.051701,645.540740,-8000.00,8483.00,-25.380,2658,276864.26,284975.68,0.971536,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100002,-9490.73,-3.813070,910.144154,-19150.00,20000.00,-25.290,2489,285876.77,295367.50,0.967868,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100003,2137.77,3.224389,748.056684,-4310.28,4409.40,-21.350,663,84178.75,82040.98,1.026057,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100004,-29837.87,-27.525710,414.375001,-1800.00,1748.00,-27.555,1084,97426.62,127264.49,0.765544,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01100005,-9756.37,-4.044930,1314.772339,-6987.07,39915.16,-25.080,2412,404922.85,414679.22,0.976472,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C01104996,38478.08,12.384319,1634.482084,-37648.00,44100.00,-26.530,3107,470767.11,432289.03,1.089010,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104997,591.59,0.512199,4459.475583,-56256.44,134384.01,-19.990,1155,217927.74,217336.15,1.002722,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,real,0.0
C01104998,-12120.38,-3.429649,394.044758,-1953.37,12710.00,-17.715,3534,136083.82,148204.20,0.918218,...,0.0,0.0,0.210526,16857.338694,1.894851,0.0,21227.966021,0.0,real,0.0


In [4]:
# Group by customer_category and FPF_TARGET, then count occurrences
category_target_counts = (
    df.groupby(["dataset_type", "FPF_TARGET"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: "count_0", 1.0: "count_1"})
)
category_target_counts

FPF_TARGET,count_0,count_1
dataset_type,,
real,3942,58


In [5]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# ========== Load and Prepare Data ==========

# Ensure masked_consumer_id is a column and string-typed
df = df.reset_index().rename(columns={'index': 'masked_consumer_id'})
df['masked_consumer_id'] = df['masked_consumer_id'].astype(str)

# Drop non-feature columns
drop_cols = ['masked_consumer_id', 'FPF_TARGET', 'dataset_type']
X = df.drop(columns=drop_cols)
y = df['FPF_TARGET']
feat_names = X.columns.tolist()

# ========== Split Data ==========

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ========== Feature Selection ==========

selector = LogisticRegression(penalty='l2', max_iter=2000, random_state=42)
selector.fit(X_train, y_train)
coefs = np.abs(selector.coef_).ravel()
top_indices = np.argsort(coefs)[::-1][:250]
selected_feats = [feat_names[i] for i in top_indices]
print(f"Selected top 250 features.")

X_train_sel = X_train.iloc[:, top_indices]
X_val_sel   = X_val.iloc[:, top_indices]

# ========== Define and Train Model ==========

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=20,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42
    ))
])

print("→ Training XGBoost on top 250 features")
pipeline.fit(X_train_sel, y_train)

# ========== Evaluate ==========

p_train = pipeline.predict_proba(X_train_sel)[:, 1]
p_val   = pipeline.predict_proba(X_val_sel)[:, 1]

train_auc = roc_auc_score(y_train, p_train)
val_auc   = roc_auc_score(y_val, p_val)

print(f"   Train AUC = {train_auc:.3f}, Val AUC = {val_auc:.3f}")

/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [20:56:36] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Selected top 250 features.
→ Training XGBoost on top 250 features
   Train AUC = 1.000, Val AUC = 0.780
